In [1]:
import torch
import pandas as pd
import re
import os
import csv
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.notebook import tqdm

# Config
BASE_MODEL     = "unsloth/Qwen2.5-3B-Instruct"
ADAPTER_PATH   = "./qwen-svg-lora/lora_adapter"
TEST_CSV       = "./csv/test.csv"
OUTPUT_CSV     = "./csv/submission_finetuned.csv"
MAX_NEW_TOKENS = 1000
BATCH_SIZE     = 8  # lower to 4 if OOM

SYSTEM_PROMPT = (
    "You are an expert SVG coder. Generate valid, complete SVG code matching "
    "the user's description. Return ONLY raw SVG code. No markdown, no explanations. "
    "The SVG must be width='256' height='256' with viewBox='0 0 256 256'. "
    "Use only allowed elements: svg, g, path, rect, circle, ellipse, line, "
    "polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, "
    "radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter. "
    "try to reduce path elements and instead use other tags like circle, rect"
    "No scripts, event handlers, animation, foreignObject, or external references."
)

FALLBACK_SVG = "<svg xmlns='http://www.w3.org/2000/svg' width='256' height='256' viewBox='0 0 256 256'><rect x='64' y='64' width='128' height='128' fill='black'/></svg>"

ALLOWED_ELEMENTS = {
    "svg","g","path","rect","circle","ellipse","line","polyline","polygon",
    "defs","use","symbol","clipPath","mask","linearGradient","radialGradient",
    "stop","text","tspan","title","desc","style","pattern","marker","filter"
}

DISALLOWED_PATTERNS = [
    r'<script', r'on\w+\s*=', r'<animate', r'<set\b',
    r'<foreignObject', r'href\s*=\s*["\']http', r'url\s*\(\s*["\']?http'
]

print("✅ Config ready")

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


✅ Config ready


In [2]:
def make_safe_svg(raw: str, row_id: str) -> str:
    raw = str(raw).strip()

    # Extract SVG block
    match = re.search(r'<svg.*?</svg>', raw, re.DOTALL | re.IGNORECASE)
    if not match:
        # Try without closing tag (truncated output)
        match = re.search(r'<svg.*', raw, re.DOTALL | re.IGNORECASE)
        if not match:
            print(f"  [WARN] {row_id}: no <svg> found, using fallback")
            return FALLBACK_SVG
        svg = match.group(0) + '</svg>'
    else:
        svg = match.group(0)

    # Strip XML comments
    svg = re.sub(r'<!--.*?-->', '', svg, flags=re.DOTALL)

    # Only strip explicitly FORBIDDEN tags
    FORBIDDEN_TAGS = ['script', 'animate', 'animateTransform', 'animateMotion',
                      'set', 'foreignObject', 'iframe', 'video', 'audio', 'canvas']
    for tag in FORBIDDEN_TAGS:
        svg = re.sub(rf'<{tag}[\s>].*?</{tag}>', '', svg, flags=re.DOTALL | re.IGNORECASE)
        svg = re.sub(rf'<{tag}\s*/>', '', svg, flags=re.IGNORECASE)

    # Remove event handlers
    svg = re.sub(r'\son\w+\s*=\s*["\'][^"\']*["\']', '', svg)
    svg = re.sub(r'href\s*=\s*["\']https?://[^"\']*["\']', '', svg)

    # Check path count
    if len(re.findall(r'<path', svg, re.IGNORECASE)) > 256:
        print(f"  [WARN] {row_id}: too many paths, using fallback")
        return FALLBACK_SVG

    # Force 256x256
    svg = re.sub(r'\bwidth\s*=\s*["\'][^"\']*["\']', "width='256'", svg, count=1)
    svg = re.sub(r'\bheight\s*=\s*["\'][^"\']*["\']', "height='256'", svg, count=1)
    if 'viewBox' not in svg:
        svg = svg.replace('<svg ', "<svg viewBox='0 0 256 256' ", 1)

    # Add xmlns if missing
    if 'xmlns=' not in svg.lower():
        svg = svg.replace('<svg', "<svg xmlns='http://www.w3.org/2000/svg'", 1)

    # Normalize whitespace, convert double quotes to single
    svg = re.sub(r'\s+', ' ', svg).strip()
    svg = svg.replace('"', "'")

    # Truncate if too long
    if len(svg) > 8000:
        print(f"  [WARN] {row_id}: truncating to 8000 chars")
        svg = svg[:8000]
        if not svg.rstrip().endswith('</svg>'):
            svg = svg.rsplit('<', 1)[0] + '</svg>'

    return svg

print("✅ Validation function ready")

✅ Validation function ready


In [3]:
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16
).to("cuda")

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.generation_config.max_length = None
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

allocated = torch.cuda.memory_allocated() / 1024**3
print(f"✅ Model ready | VRAM: {allocated:.2f} GB used")

Loading base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter...
✅ Model ready | VRAM: 5.97 GB used


In [4]:
df = pd.read_csv(TEST_CSV)
print(f"Loaded {len(df)} prompts")

# Resume from existing output if it exists
results = {}
if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    results = dict(zip(existing["id"], existing["svg"]))
    print(f"Resuming — {len(results)} done, {len(df) - len(results)} remaining")
else:
    print("Starting fresh")

pending = df[~df["id"].isin(results.keys())].reset_index(drop=True)
print(f"To generate: {len(pending)}")

Loaded 1000 prompts
Starting fresh
To generate: 1000


In [8]:
# Quick test with 20 prompts
test_sample = pending.iloc[0:20].reset_index(drop=True)
sample_results = {}

print(f"🧪 Testing with 20 prompts (batch size={BATCH_SIZE})...\n")

for i in tqdm(range(0, len(test_sample), BATCH_SIZE)):
    batch = test_sample.iloc[i:i + BATCH_SIZE]
    prompts = [build_prompt(p) for p in batch["prompt"].tolist()]
    ids = batch["id"].tolist()

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens      = MAX_NEW_TOKENS,
            temperature         = 0.2,    # up from 0.1
            min_p               = 0.1,    # add this
            do_sample           = True,
            pad_token_id        = tokenizer.pad_token_id,
            eos_token_id        = terminators,
            repetition_penalty  = 1.3,
        )

    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

    for row_id, raw in zip(ids, decoded):
        sample_results[row_id] = make_safe_svg(raw, row_id)

# Summary
fallbacks = sum(1 for v in sample_results.values() if v == FALLBACK_SVG)
print(f"\n✅ Done!")
print(f"  Total: {len(sample_results)}")
print(f"  Fallbacks: {fallbacks}")
print(f"  Valid SVGs: {len(sample_results) - fallbacks}")
print(f"\nSample output:\n{list(sample_results.values())[0][:300]}...")

🧪 Testing with 20 prompts (batch size=8)...



  0%|          | 0/3 [00:00<?, ?it/s]


✅ Done!
  Total: 20
  Fallbacks: 0
  Valid SVGs: 20

Sample output:
<svg xmlns='http://www.w3.org/2000/svg' enable-background='new 0 0 193.847 193.847' height='256' viewBox='-1 -1 193.847 193.847' width='256' xmlns:xlink='' xlink:href='#_xlink'><path d='m-1-.001c-1.11 0-2 .9-2 2v193h-192z'/><g fill-rule='evenodd' transform='translate(1)'><circle cx='.001' cy='.001' ...


In [9]:
for i in list(sample_results.values()):
    print(i)

<svg xmlns='http://www.w3.org/2000/svg' enable-background='new 0 0 193.847 193.847' height='256' viewBox='-1 -1 193.847 193.847' width='256' xmlns:xlink='' xlink:href='#_xlink'><path d='m-1-.001c-1.11 0-2 .9-2 2v193h-192z'/><g fill-rule='evenodd' transform='translate(1)'><circle cx='.001' cy='.001' r='1'/></g></svg>
<svg xmlns='http://www.w3.org/2000/svg' version='1.1' id='_id_4879_' class='' x='0px' y='0px' width='256' height='256' viewBox='-1 -1 256 256'> <path fill='#FFFFFF' d='M-1 -1 L256 256'/> </path> <g transform=''> <path fill='#FFDABF' stroke-width='.5' opacity='1' d='M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z'></path> <path fill='#EFCBAC' stroke-width='.5' opacity='1' d='M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z'></path> <path fill='#CDBAAD' stroke-width='.5' opacity='1' d='M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z M-1 -.5 C-.5 .5 -1 .5 -1 .5 H256 V256 Z'></path> <path fill='#FFFDFD' stroke-width='.5' op

In [15]:
# Debug the failing ones
failing_ids = [
    'fa1d8fa7-080f-4269-a9cf-a17562c9a0ca',
    '6eede943219547c22ac56085027d33cc',
    '8fe82f3af89e487b31236ca829c3f071',
]

failing = pending[pending['id'].isin(failing_ids)].reset_index(drop=True)
prompts = [build_prompt(p) for p in failing["prompt"].tolist()]

inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512,
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
    )

input_len = inputs["input_ids"].shape[1]
decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

for i, (row_id, raw) in enumerate(zip(failing_ids, decoded)):
    print(f"=== {row_id} ===")
    print(f"PROMPT: {failing.iloc[i]['prompt']}")
    print(f"RAW OUTPUT: {repr(raw[:300])}")
    print()

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== fa1d8fa7-080f-4269-a9cf-a17562c9a0ca ===
PROMPT: firewood stack cut logs wood with leaf illustration.
RAW OUTPUT: '<svg enable-background="new 0 0 256 256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg"><path d="m174.8 19.2c-1.6-1.6-4.8-1.6-6.4 0l-16.8 16.8c-1.6 1.6-1.6 4.8 0 6.4s4.8 1.6 6.4 0l16.8-16.8c1.6-1.6 4.8-1.6 6.4 0s1.6 4.8 0 6.4z"/><path d="m174.8 19.2c-1.6-1.6-4.8-1.6-6.4 0l-16.8 16.8c-1.6 1'

=== 6eede943219547c22ac56085027d33cc ===
PROMPT: The image shows five horizontal lines of varying thicknesses and lengths, arranged vertically on a white background.
RAW OUTPUT: '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256" width="256" height="256"><path fill="#444444" fill-opacity="1.0"  filling="0" d="M178.93800354003906 12.5 L178.93800354003906 24.061996459960938 L178.93800354003906 25.625 C178.93800354003906 26.25 178.93800354003906 26.25 178.93800354003'

=== 8fe82f3af89e487b31236ca829c3f071 ===
PROMPT: The image contains black geometric shapes again

In [5]:
def build_prompt(prompt: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nGenerate SVG for: {prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def save_results(results, path):
    out = pd.DataFrame([{"id": k, "svg": v} for k, v in results.items()])
    out.to_csv(path, index=False, quoting=csv.QUOTE_ALL)
    print(f"  💾 Saved {len(results)} rows → {path}")


terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]


In [ ]:


print(f"🚀 Generating {len(pending)} SVGs (batch size={BATCH_SIZE})...\n")

for i in tqdm(range(0, len(pending), BATCH_SIZE)):
    batch = pending.iloc[i:i + BATCH_SIZE]
    prompts = [build_prompt(p) for p in batch["prompt"].tolist()]
    ids = batch["id"].tolist()

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False
    ).to("cuda")

    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=terminators,
            )

        input_len = inputs["input_ids"].shape[1]
        decoded = tokenizer.batch_decode(
            outputs[:, input_len:], skip_special_tokens=True
        )

        for row_id, raw in zip(ids, decoded):
            results[row_id] = make_safe_svg(raw, row_id)

    except Exception as e:
        print(f"  [ERROR] batch {i}: {e} — using fallback")
        for row_id in ids:
            results[row_id] = FALLBACK_SVG

    # Checkpoint every 100 rows
    if (i + BATCH_SIZE) % 100 == 0:
        save_results(results, OUTPUT_CSV)

# Final save
save_results(results, OUTPUT_CSV)
print(f"\n✅ All done! Submission at {OUTPUT_CSV}")

🚀 Generating 1000 SVGs (batch size=8)...



  0%|          | 0/125 [00:00<?, ?it/s]

Both `max_new_tokens` (=1000) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1000) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [WARN] fa1d8fa7-080f-4269-a9cf-a17562c9a0ca: no <svg> found, using fallback
  [WARN] 6eede943219547c22ac56085027d33cc: no <svg> found, using fallback
  [WARN] ea045c7a247166f061ce504d9b7ccaab: no <svg> found, using fallback
  [WARN] 8fe82f3af89e487b31236ca829c3f071: no <svg> found, using fallback
  [WARN] 600464e4d92c75338462271a09b3f176: no <svg> found, using fallback
  [WARN] 8cbefcd53fd5dfa2cbe1b374356ed709: no <svg> found, using fallback
  [WARN] 625181a2-600d-4db5-83c6-1a34676eb0dc: no <svg> found, using fallback


KeyboardInterrupt: 